# Agriculture Crop Production Prediction in India
**Domain:** Data Science & Machine Learning  
**Week 04 – Final Project**


## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_squared_error
import pickle
import warnings
warnings.filterwarnings('ignore')
print('Libraries loaded ✅')

## 2. Load Dataset

In [ ]:
df = pd.read_csv('../data/crop_data.csv')
print(f'Shape: {df.shape}')
df.head()

In [ ]:
df.describe()

## 3. Data Preprocessing

In [ ]:
# Clean column names
df.columns = df.columns.str.strip().str.replace(' ', '_')
print('Missing values:\n', df.isnull().sum())
df.dropna(inplace=True)

# Encode categoricals
le_crop   = LabelEncoder()
le_state  = LabelEncoder()
le_season = LabelEncoder()

df['Crop_encoded']   = le_crop.fit_transform(df['Crop'])
df['State_encoded']  = le_state.fit_transform(df['State'])
df['Season_encoded'] = le_season.fit_transform(df['Season'])
print('Encoding done ✅')

## 4. Exploratory Data Analysis

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('EDA — Agriculture Crop Data', fontsize=16)

axes[0,0].hist(df['Yield'], bins=40, color='steelblue', edgecolor='white')
axes[0,0].set_title('Yield Distribution')

crop_yield = df.groupby('Crop')['Yield'].mean().sort_values(ascending=False)
axes[0,1].barh(crop_yield.index, crop_yield.values, color='seagreen')
axes[0,1].set_title('Avg Yield by Crop')

axes[1,0].scatter(df['Cost_of_Cultivation'], df['Yield'], alpha=0.3, s=8, color='coral')
axes[1,0].set_title('Cost vs Yield')

num_cols = ['Cost_of_Cultivation','Cost_of_Production','Annual_Rainfall','Temperature','Yield']
sns.heatmap(df[num_cols].corr(), annot=True, fmt='.2f', cmap='coolwarm', ax=axes[1,1])
axes[1,1].set_title('Correlation Heatmap')

plt.tight_layout()
plt.show()

## 5. Model Training

In [ ]:
features = ['Crop_encoded','State_encoded','Season_encoded','Area',
            'Cost_of_Cultivation','Cost_of_Production','Annual_Rainfall','Temperature']
X = df[features]
y = df['Yield']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

lr = LinearRegression()
lr.fit(X_train, y_train)

rf = RandomForestRegressor(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)

print('Models trained ✅')

## 6. Model Evaluation

In [ ]:
for name, model in [('Linear Regression', lr), ('Random Forest', rf)]:
    preds = model.predict(X_test)
    print(f"{name}")
    print(f"  R²   : {r2_score(y_test, preds):.4f}")
    print(f"  MSE  : {mean_squared_error(y_test, preds):,.0f}")
    print(f"  RMSE : {np.sqrt(mean_squared_error(y_test, preds)):,.0f}\n")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, (name, m) in zip(axes, [('Linear Regression', lr), ('Random Forest', rf)]):
    p = m.predict(X_test)
    ax.scatter(y_test, p, alpha=0.3, s=10)
    lims = [y_test.min(), y_test.max()]
    ax.plot(lims, lims, 'r--')
    ax.set_title(f'{name} — R²={r2_score(y_test, p):.4f}')
    ax.set_xlabel('Actual Yield')
    ax.set_ylabel('Predicted Yield')
plt.tight_layout()
plt.show()

## 7. Feature Importance

In [ ]:
imp = pd.Series(rf.feature_importances_, index=features).sort_values()
imp.plot.barh(figsize=(8,5), color='darkorange')
plt.title('Feature Importances — Random Forest')
plt.tight_layout()
plt.show()

## 8. Make a Prediction

In [ ]:
sample = pd.DataFrame([{
    'Crop_encoded':   le_crop.transform(['Rice'])[0],
    'State_encoded':  le_state.transform(['Punjab'])[0],
    'Season_encoded': le_season.transform(['Kharif'])[0],
    'Area': 100,
    'Cost_of_Cultivation': 35000,
    'Cost_of_Production':  25000,
    'Annual_Rainfall': 900,
    'Temperature': 28
}])

pred_yield = rf.predict(sample)[0]
print(f'Predicted Yield for Rice in Punjab (Kharif): {pred_yield:.2f} kg/ha')